# 1. Ransomware Resiliency — Architecture Design

SC-100 doesn't ask "what is ransomware?" — it asks **"design a resiliency strategy for Contoso that protects their critical assets and ensures business continuity."**

## Microsoft's ransomware protection framework

```
┌─────────────────────────────────────────────────────────────────┐
│                    PREPARE (before attack)                      │
│  ┌──────────┐  ┌──────────────┐  ┌───────────────────────────┐ │
│  │ Identify  │  │ Protect      │  │ Backup & Recovery         │ │
│  │ critical  │  │ privileged   │  │ Immutable backups         │ │
│  │ assets    │  │ access       │  │ Cross-region replication  │ │
│  └──────────┘  └──────────────┘  │ Offline backup copy       │ │
│                                   │ Tested restore procedures │ │
│                                   └───────────────────────────┘ │
├─────────────────────────────────────────────────────────────────┤
│                    DETECT (during attack)                       │
│  Defender for Endpoint → Sentinel → XDR correlation             │
│  Attack disruption → automated containment                     │
├─────────────────────────────────────────────────────────────────┤
│                    RECOVER (after attack)                       │
│  BCDR playbook → restore from immutable backups                │
│  Rebuild compromised identities → rotate all secrets           │
│  Post-incident review → improve detections                     │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
import json

# ===================================================================
# SCENARIO: Design a ransomware resiliency strategy for Contoso
# ===================================================================

SCENARIO = """
COMPANY: Contoso Financial Services
EMPLOYEES: 5,000
INFRASTRUCTURE:
  - Azure: 3 subscriptions (prod, staging, dev)
  - On-premises: Active Directory, legacy financial apps
  - Microsoft 365: Exchange Online, SharePoint, Teams
  - 200 VMs across Azure and on-prem
  - Azure SQL databases with customer financial data
  - Storage accounts with regulatory documents

CURRENT STATE:
  - Azure Backup configured but never tested
  - No immutable backups
  - Global admin accounts don't use PIM
  - Defender for Endpoint on 60% of devices
  - No Sentinel deployment
  - Conditional Access: MFA for admins only
  - Network: flat network, no segmentation

REQUIREMENT: Board mandates ransomware readiness within 90 days after
industry peer suffered $50M ransomware incident.
"""

print(SCENARIO)

In [ ]:
# Step 1: Identify and prioritize critical assets
# An architect's first job is knowing WHAT to protect

CRITICAL_ASSETS = [
    {'asset': 'Customer financial database (Azure SQL)',       'impact': 'Critical', 'rto': '1 hour',  'rpo': '15 min', 'reason': 'Regulatory requirement, customer trust'},
    {'asset': 'Active Directory / Entra ID',                  'impact': 'Critical', 'rto': '2 hours', 'rpo': '0',       'reason': 'All auth depends on this, total lockout if compromised'},
    {'asset': 'Microsoft 365 (Exchange, SharePoint)',          'impact': 'High',     'rto': '4 hours', 'rpo': '1 hour',  'reason': 'Business communication, document collaboration'},
    {'asset': 'Regulatory document storage',                  'impact': 'High',     'rto': '8 hours', 'rpo': '24 hours', 'reason': 'Compliance, but rarely changes'},
    {'asset': 'Legacy financial applications (on-prem VMs)',  'impact': 'High',     'rto': '4 hours', 'rpo': '1 hour',  'reason': 'Core business processes'},
    {'asset': 'Development environment',                      'impact': 'Medium',   'rto': '24 hours','rpo': '24 hours', 'reason': 'Can rebuild from git, low urgency'},
]

print('=== Step 1: Critical Asset Inventory ===\n')
print(f'{"Asset":<50} {"Impact":<10} {"RTO":<10} {"RPO":<10}')
print('─' * 80)
for a in CRITICAL_ASSETS:
    print(f'{a["asset"]:<50} {a["impact"]:<10} {a["rto"]:<10} {a["rpo"]:<10}')

print('\n💡 RTO = Recovery Time Objective (how fast must it be back?)')
print('   RPO = Recovery Point Objective (how much data loss is acceptable?)')

In [ ]:
# Step 2: Design the BCDR strategy

BCDR_DESIGN = {
    'Azure SQL (Critical)': {
        'backup': 'Automated backups with long-term retention (LTR)',
        'replication': 'Active geo-replication to paired region',
        'immutability': 'Immutable backup vault with resource lock',
        'testing': 'Monthly restore drill to isolated environment',
        'meets_rto': '1 hour via failover group',
        'meets_rpo': '< 5 seconds with active geo-replication',
    },
    'Entra ID / AD DS (Critical)': {
        'backup': 'Entra ID: soft-delete + recycle bin (30 days). AD DS: system state backup daily.',
        'replication': 'Entra ID: built-in global replication. AD DS: 2+ DCs per domain.',
        'immutability': 'Break-glass accounts stored in physical safe + separate cloud-only admin',
        'testing': 'Quarterly AD restore to isolated forest',
        'key_action': 'PIM for all admin roles — limits blast radius if admin account is compromised',
    },
    'Storage Accounts (High)': {
        'backup': 'Azure Backup with immutable vault',
        'replication': 'GRS (Geo-Redundant Storage)',
        'immutability': 'Immutable blob storage with legal hold',
        'versioning': 'Blob versioning + soft delete (14 days)',
    },
    'VMs (High)': {
        'backup': 'Azure Backup with immutable recovery services vault',
        'replication': 'Azure Site Recovery for critical VMs',
        'testing': 'ASR test failover monthly (doesn\'t affect production)',
    },
    'M365 (High)': {
        'backup': 'Microsoft 365 Backup (native) + third-party backup for SharePoint/Exchange',
        'retention': 'Retention policies: 7 years for email, 3 years for SharePoint',
    },
}

print('=== Step 2: BCDR Architecture Design ===\n')
for asset, design in BCDR_DESIGN.items():
    print(f'📦 {asset}')
    for k, v in design.items():
        print(f'   {k}: {v}')
    print()

In [ ]:
# Step 3: Prioritized 90-day implementation plan

PHASES = [
    {
        'phase': 'Phase 1: Weeks 1-2 (QUICK WINS)',
        'actions': [
            ('Enable PIM for all admin roles', 'Identity', 'Critical — limits blast radius immediately'),
            ('MFA for ALL users (not just admins)', 'Identity', 'Blocks 99.9% of identity attacks'),
            ('Enable immutable backup vaults', 'Backup', 'Prevents ransomware from deleting backups'),
            ('Deploy Defender for Endpoint to remaining 40%', 'Endpoint', 'Detection coverage gap'),
            ('Block legacy authentication via CA', 'Identity', 'Legacy protocols can\'t do MFA'),
        ],
    },
    {
        'phase': 'Phase 2: Weeks 3-6 (CORE PROTECTION)',
        'actions': [
            ('Deploy Microsoft Sentinel', 'Detection', 'Central SIEM for correlation'),
            ('Enable all Defender for Cloud plans (Servers P2, SQL, Storage)', 'Detection', 'Workload protection'),
            ('Implement network segmentation (NSGs + Azure Firewall)', 'Network', 'Limit lateral movement'),
            ('Configure Azure SQL active geo-replication', 'BCDR', 'Meet 1-hour RTO for critical DB'),
            ('Deploy ASR for critical on-prem VMs', 'BCDR', 'Failover capability'),
            ('Create break-glass admin accounts', 'Identity', 'Emergency access if Entra ID is attacked'),
        ],
    },
    {
        'phase': 'Phase 3: Weeks 7-12 (MATURE & TEST)',
        'actions': [
            ('Run BCDR drill — full restore test', 'BCDR', 'Verify backups actually work'),
            ('Create ransomware incident response playbook', 'SecOps', 'Documented procedures'),
            ('Tabletop exercise with leadership', 'SecOps', 'Test decision-making under pressure'),
            ('Implement Conditional Access: risk-based + device compliance', 'Identity', 'Zero Trust hardening'),
            ('Configure Sentinel analytics rules for ransomware TTPs', 'Detection', 'Proactive detection'),
            ('Review and harden AD DS (tier model, GPO audit)', 'Identity', 'On-prem identity protection'),
        ],
    },
]

print('=== Step 3: 90-Day Ransomware Readiness Plan ===\n')
for phase in PHASES:
    print(f'\n📅 {phase["phase"]}')
    print('─' * 60)
    for action, category, reason in phase['actions']:
        print(f'  ☐ [{category}] {action}')
        print(f'     Why: {reason}')

## Architecture decision: where the exam tests you

The exam presents a scenario and asks you to **choose the right approach**:

| Question pattern | What they're testing |
|-----------------|---------------------|
| "Contoso needs to protect backups from ransomware" | **Immutable backup vault** + resource locks |
| "Ensure critical DB recovers within 1 hour" | **Active geo-replication** + failover groups (not just backup) |
| "Limit blast radius of compromised admin" | **PIM** + privileged access workstations + network segmentation |
| "Which should be prioritized first?" | Identity protections (MFA + PIM) before network |
| "What should the BCDR test include?" | Full restore test, not just backup verification |

### Key principle: Microsoft's recommended priority order

1. **Protect identities** (MFA, PIM, break-glass) — most attacks start here
2. **Protect backups** (immutable, cross-region, tested) — your recovery lifeline
3. **Detect & respond** (Sentinel, Defender XDR) — reduce dwell time
4. **Segment networks** (NSG, Firewall) — limit lateral movement
5. **Harden endpoints** (Defender for Endpoint, ASR rules) — reduce attack surface

**Next**: [Notebook 2 — Frameworks and Zero Trust](02_frameworks_and_zero_trust.ipynb)